# Tarea 4 — Prueba de transcripción (Colab, GPU)

Transcribe los audios que subas con `faster-whisper` (modelo `medium`, `float16`, español) y mide, por audio, el dispositivo, la VRAM y el tiempo.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4).

## 1. Instalar dependencias

In [ ]:
# Colab ya trae torch; pip solo lo actualiza si hace falta.
# Si más adelante falla con un error de libcudnn / cublas, instala también: !pip install -q nvidia-cudnn-cu12
!pip install -q faster-whisper torch

## 2. Imports y verificación de GPU

In [ ]:
import json
import threading
import time
from pathlib import Path

import torch
from faster_whisper import WhisperModel
from google.colab import files

# Esta prueba solo tiene sentido en GPU: si no hay, avisamos en vez de caer a CPU en silencio.
if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU. Activa una en: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU."
    )
print("GPU:", torch.cuda.get_device_name(0))

## 3. Cargar el modelo (medium, float16, español)

In [ ]:
MODEL_SIZE = "medium"
COMPUTE_TYPE = "float16"
LANGUAGE = "es"

# Línea base de VRAM usada en toda la GPU, antes de cargar el modelo (mem_get_info también inicializa CUDA en torch).
libre, total = torch.cuda.mem_get_info()
vram_base_mb = (total - libre) / 1024**2

model = WhisperModel(MODEL_SIZE, device="cuda", compute_type=COMPUTE_TYPE)

# Dispositivo que realmente usa el modelo (CTranslate2); debe decir "cuda".
device_used = str(getattr(model.model, "device", "cuda"))
print("Dispositivo del modelo:", device_used)

## 4. Subir los audios

Selecciona los archivos de audio (por ejemplo `entrevista_01.wav`, `llamada_01.wav`).

In [ ]:
subidos = files.upload()  # dict {nombre_archivo: bytes}; Colab los guarda en el directorio actual

EXTENSIONES = {".wav", ".mp3", ".ogg", ".opus", ".m4a", ".flac"}
audios = [n for n in subidos if Path(n).suffix.lower() in EXTENSIONES]
omitidos = [n for n in subidos if n not in audios]
print(f"{len(audios)} audio(s) para transcribir.")
if omitidos:
    print("Omitidos (no son audio):", omitidos)

## 5. Función de transcripción y medición

- `vram_mb` usa `torch.cuda.max_memory_allocated()`, como se pidió. **Ojo:** faster-whisper corre sobre CTranslate2, que reserva su propia memoria de GPU fuera del allocator de PyTorch, así que ese valor suele salir ~0.
- Por eso también se guarda `vram_device_mb`: el pico de memoria usada en toda la GPU durante el audio (muestreado con `torch.cuda.mem_get_info()`), restando la línea base previa a cargar el modelo. Incluye los pesos del modelo.

In [ ]:
class MuestreadorVram:
    """Muestrea en un hilo aparte la memoria usada de toda la GPU y guarda el máximo (MB)."""

    def __init__(self, intervalo=0.05):
        self.intervalo = intervalo
        self.pico_mb = 0.0

    def _muestrear(self):
        libre, total = torch.cuda.mem_get_info()
        self.pico_mb = max(self.pico_mb, (total - libre) / 1024**2)

    def _correr(self):
        while not self._parar.is_set():
            self._muestrear()
            time.sleep(self.intervalo)

    def __enter__(self):
        self._parar = threading.Event()
        self._hilo = threading.Thread(target=self._correr, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *exc):
        self._parar.set()
        self._hilo.join()
        self._muestrear()  # última muestra, por si el pico ocurrió entre dos ciclos


def transcribir(nombre_archivo):
    """Transcribe un audio y devuelve el dict con las métricas."""
    torch.cuda.reset_peak_memory_stats()
    with MuestreadorVram() as vram:
        t0 = time.perf_counter()
        segmentos, _ = model.transcribe(nombre_archivo, language=LANGUAGE, beam_size=5)
        # `segmentos` es un generador: la transcripción ocurre al recorrerlo, por eso se une dentro del cronómetro.
        texto = " ".join(seg.text.strip() for seg in segmentos)
        torch.cuda.synchronize()
        transcurrido = time.perf_counter() - t0

    return {
        "clip": Path(nombre_archivo).stem,
        "transcript": texto,
        "device": device_used,
        "vram_mb": round(torch.cuda.max_memory_allocated() / 1024**2, 1),
        "elapsed_s": round(transcurrido, 2),
        "vram_device_mb": round(vram.pico_mb - vram_base_mb, 1),
    }

## 6. Transcribir todos los audios

El primer audio suele tardar más (calentamiento de CUDA); compara los siguientes para ver el tiempo real.

In [ ]:
resultados = {}  # {clip: dict con clip, transcript, device, vram_mb, elapsed_s, vram_device_mb}

for nombre in audios:
    r = transcribir(nombre)
    resultados[r["clip"]] = r

    # Un JSON por audio, para commitear como caché de transcripciones
    Path(f"{r['clip']}.json").write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"=== {r['clip']} ===")
    print("Texto:", r["transcript"])
    print("Dispositivo:", r["device"])
    print(f"VRAM peak (torch.cuda.max_memory_allocated): {r['vram_mb']} MB")
    print(f"VRAM peak en toda la GPU (incluye modelo): {r['vram_device_mb']} MB")
    print(f"Tiempo: {r['elapsed_s']} s\n")

## 7. Descargar los JSON

In [ ]:
# El navegador puede pedir permiso para descargar varios archivos seguidos: acéptalo.
for clip in sorted(resultados):
    files.download(f"{clip}.json")